In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import ast
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
from sklearn.feature_selection import RFE
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV, GroupShuffleSplit, RandomizedSearchCV, cross_val_score, KFold
from shapely.geometry import LineString, Point
from tqdm import tqdm
from itertools import combinations
from sklearn.neighbors import BallTree
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer, PolynomialFeatures, RobustScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor, plot_importance, DMatrix, train
from lightgbm import LGBMRegressor, early_stopping
import shap
from scipy.stats import zscore
from scipy.spatial import cKDTree
import os
import fiona
import random
import pickle
from geopy.distance import geodesic

/Users/silarbinunzio/opt/anaconda3/lib/python3.8/site-packages/dask/dataframe/utils.py:367: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/Users/silarbinunzio/opt/anaconda3/lib/python3.8/site-packages/dask/dataframe/utils.py:367: FutureWarning: pandas.Float64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/Users/silarbinunzio/opt/anaconda3/lib/python3.8/site-packages/dask/dataframe/utils.py:367: FutureWarning: pandas.UInt64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)


In [2]:
import array_fet

In [3]:
firepoint = pd.read_pickle("../data/firefrance/df_full_departement_None_node.pkl")

In [4]:
train_features = [
    'temp', 'dwpt', 'rhum', 'prcp', 'wdir', 'wspd', 'prec24h',
    'dc', 'ffmc', 'dmc', 'nesterov', 'munger', 'kbdi', 'isi', 'angstroem', 'bui',
    'fwi', 'dailySeverityRating', 'temp16', 'dwpt16', 'rhum16', 'prcp16', 'wdir16',
    'wspd16', 'prec24h16', 'days_since_rain', 'sum_consecutive_rainfall', 'sum_rain_last_7_days',
    'sum_snow_last_7_days', 'snow24h', 'snow24h16', 'elevation', 'population',
    'foret_encoder', 'argile_encoder', 'id_encoder', 'cluster_encoder', 'cosia_encoder',
    'vigicrues', 'foret', 'highway', 'cosia', 'Calendar', 'Geo', 'nappes', 'AutoRegressionBin'
]

METHODS_SPATIAL_TRAIN = ['mean', 'sum', 'std']

In [5]:
features_name, _ = array_fet.get_features_name_list('departement', train_features, METHODS_SPATIAL_TRAIN)

print("Liste des features :", features_name)

Liste des features : ['temp_mean', 'temp_sum', 'temp_std', 'dwpt_mean', 'dwpt_sum', 'dwpt_std', 'rhum_mean', 'rhum_sum', 'rhum_std', 'prcp_mean', 'prcp_sum', 'prcp_std', 'wdir_mean', 'wdir_sum', 'wdir_std', 'wspd_mean', 'wspd_sum', 'wspd_std', 'prec24h_mean', 'prec24h_sum', 'prec24h_std', 'dc_mean', 'dc_sum', 'dc_std', 'ffmc_mean', 'ffmc_sum', 'ffmc_std', 'dmc_mean', 'dmc_sum', 'dmc_std', 'nesterov_mean', 'nesterov_sum', 'nesterov_std', 'munger_mean', 'munger_sum', 'munger_std', 'kbdi_mean', 'kbdi_sum', 'kbdi_std', 'isi_mean', 'isi_sum', 'isi_std', 'angstroem_mean', 'angstroem_sum', 'angstroem_std', 'bui_mean', 'bui_sum', 'bui_std', 'fwi_mean', 'fwi_sum', 'fwi_std', 'dailySeverityRating_mean', 'dailySeverityRating_sum', 'dailySeverityRating_std', 'temp16_mean', 'temp16_sum', 'temp16_std', 'dwpt16_mean', 'dwpt16_sum', 'dwpt16_std', 'rhum16_mean', 'rhum16_sum', 'rhum16_std', 'prcp16_mean', 'prcp16_sum', 'prcp16_std', 'wdir16_mean', 'wdir16_sum', 'wdir16_std', 'wspd16_mean', 'wspd16_sum',

In [13]:
firepoint.head()

,graph_id,id,longitude,latitude,departement,date,weight,days_until_next_event,nbsinister_id,class_risk,...,dwpt16_max_max_7,rhum16_mean_max_7,rhum16_min_max_7,rhum16_max_max_7,wdir16_mean_max_7,wdir16_min_max_7,wdir16_max_max_7,wspd16_mean_max_7,wspd16_min_max_7,wspd16_max_max_7
0,0.0,0.0,5.349909,46.098569,1.0,0.0,1.0,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,0.0,5.349909,46.098569,1.0,1.0,1.0,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,0.0,5.349909,46.098569,1.0,2.0,1.0,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0,0.0,5.349909,46.098569,1.0,3.0,1.0,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0,0.0,5.349909,46.098569,1.0,4.0,1.0,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
